# 05 — Evaluate E5, BM25 và Hybrid Retriever

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
!pip install bm25s

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.5/74.5 kB 4.9 MB/s eta 0:00:00


In [28]:
from pathlib import Path
import gc, json, math, time
import bm25s, numpy as np, pandas as pd, torch
from sentence_transformers import SentenceTransformer

ROOT = Path("/content/drive/MyDrive/UIT_Legal_System")
PROCESSED = ROOT / "Dataset/processed"
TRAINED = ROOT / "Source Code/Embedding Model/multilingual_e5_base_finetuned_ver2/checkpoints/checkpoint-484"
OUTPUT = ROOT / "Source Code/Embedding Model/retriever_evaluation_e5"
OUTPUT.mkdir(parents=True, exist_ok=True)

BASE = "intfloat/multilingual-e5-base"
CORPUS = pd.read_parquet(PROCESSED / "corpus.parquet").reset_index(drop=True)
PASSAGE_IDS = CORPUS["passage_id"].astype(str).tolist()

SELECTION_SPLITS = ["val", "strict_val"]
TEST_SPLITS = ["test", "strict_test"]
CORPUS_FIELDS = ["content", "embedding_text"]
TOP_KS = [1, 3, 5, 10, 20]
MAX_K = 20
RRF_KS = [30, 60, 90]
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.bfloat16 if torch.cuda.is_available() else torch.float32

In [29]:
def fq(x): return f"query: {str(x).strip()}"
def fp(x): return f"passage: {str(x).strip()}"

def load_split(name):
    with (PROCESSED / f"{name}_queries.json").open("r", encoding="utf-8") as f:
        q = json.load(f)
    with (PROCESSED / f"{name}_qrels.json").open("r", encoding="utf-8") as f:
        r = json.load(f)
    return q, r

def load_model(ref):
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    m = SentenceTransformer(str(ref), device=DEVICE, model_kwargs={"torch_dtype": DTYPE})
    m.max_seq_length = 512
    return m

In [30]:
bm25_texts = CORPUS["embedding_text"].fillna("").astype(str).tolist()
bm25_tokens = bm25s.tokenize(bm25_texts, stopwords=None, stemmer=None, show_progress=True)
BM25 = bm25s.BM25(method="lucene")
BM25.index(bm25_tokens, show_progress=True)

Split strings:   0%|          | 0/1045 [00:00<?, ?it/s]

DEBUG:bm25s:Building index from IDs objects


BM25S Count Tokens:   0%|          | 0/1045 [00:00<?, ?it/s]

BM25S Compute Scores:   0%|          | 0/1045 [00:00<?, ?it/s]

In [31]:
def dense_rank(qemb, pemb, k):
    scores = qemb @ pemb.T
    idx = np.argsort(-scores, axis=1)[:, :k]
    return idx

def bm25_rank(questions, k):
    qt = bm25s.tokenize(questions, stopwords=None, stemmer=None, show_progress=True)
    idx, _ = BM25.retrieve(qt, k=k, show_progress=True)
    return idx

def rrf(dense_idx, sparse_idx, rrf_k, k):
    rows = []
    for drow, srow in zip(dense_idx, sparse_idx):
        scores = {}
        for rank, i in enumerate(drow, 1):
            scores[int(i)] = scores.get(int(i), 0.0) + 1/(rrf_k+rank)
        for rank, i in enumerate(srow, 1):
            scores[int(i)] = scores.get(int(i), 0.0) + 1/(rrf_k+rank)
        rows.append([i for i,_ in sorted(scores.items(), key=lambda x:x[1], reverse=True)[:k]])
    return np.asarray(rows)

def dcg(rels, k):
    return sum(rel/math.log2(rank+1) for rank, rel in enumerate(rels[:k], 1))

def metrics(query_ids, ranked_idx, qrels):
    recall = {k:0.0 for k in TOP_KS}
    rr, ndcg, ap = [], [], []
    for row, qid in enumerate(query_ids):
        relevant = set(map(str, qrels[qid]))
        ranked = [PASSAGE_IDS[int(i)] for i in ranked_idx[row]]
        rels = [1 if pid in relevant else 0 for pid in ranked]
        for k in TOP_KS:
            recall[k] += sum(rels[:k]) / max(len(relevant),1)
        first = next((r for r,x in enumerate(rels[:10],1) if x), None)
        rr.append(0 if first is None else 1/first)
        ideal = [1]*min(len(relevant),10)
        ndcg.append(dcg(rels,10)/(dcg(ideal,10) or 1))
        hits = 0; ps = 0.0
        for rank, rel in enumerate(rels[:10],1):
            if rel:
                hits += 1; ps += hits/rank
        ap.append(ps/min(len(relevant),10) if relevant else 0)
    out = {f"Recall@{k}": recall[k]/len(query_ids) for k in TOP_KS}
    out.update({"MRR@10":float(np.mean(rr)),"nDCG@10":float(np.mean(ndcg)),"MAP@10":float(np.mean(ap))})
    return out

In [32]:
def evaluate_model(model_key, model_ref, splits):
    model = load_model(model_ref)
    results = []
    for field in CORPUS_FIELDS:
        passages = [fp(x) for x in CORPUS[field].fillna("").astype(str)]
        pstart = time.perf_counter()
        pemb = model.encode(passages, batch_size=32, normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=True).astype(np.float32)
        psec = time.perf_counter()-pstart

        for split in splits:
            qraw, qrels = load_split(split)
            qids = [qid for qid in qraw if qid in qrels]
            questions = [qraw[qid] for qid in qids]
            qstart = time.perf_counter()
            qemb = model.encode([fq(x) for x in questions], batch_size=64, normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=True).astype(np.float32)
            qsec = time.perf_counter()-qstart

            didx = dense_rank(qemb, pemb, MAX_K)
            bidx = bm25_rank(questions, MAX_K)

            results.append({"model_key":model_key,"retriever":"dense","split":split,"corpus_field":field,"rrf_k":None,"corpus_encode_seconds":psec,"query_encode_seconds":qsec,**metrics(qids,didx,qrels)})
            results.append({"model_key":"bm25","retriever":"bm25","split":split,"corpus_field":"embedding_text","rrf_k":None,"corpus_encode_seconds":0.0,"query_encode_seconds":0.0,**metrics(qids,bidx,qrels)})

            for k in RRF_KS:
                hidx = rrf(didx,bidx,k,MAX_K)
                results.append({"model_key":model_key,"retriever":"hybrid_rrf","split":split,"corpus_field":field,"rrf_k":k,"corpus_encode_seconds":psec,"query_encode_seconds":qsec,**metrics(qids,hidx,qrels)})
    del model
    return results

In [33]:
selection = []
selection += evaluate_model("multilingual_e5_base", BASE, SELECTION_SPLITS)
selection += evaluate_model("multilingual_e5_finetuned", TRAINED, SELECTION_SPLITS)

selection_df = pd.DataFrame(selection).drop_duplicates(
    subset=["model_key","retriever","split","corpus_field","rrf_k"]
)
selection_df.to_csv(OUTPUT/"selection_results.csv", index=False, encoding="utf-8-sig")
display(selection_df.sort_values(["split","Recall@10","MRR@10"], ascending=[True,False,False]))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/33 [00:00<?, ?it/s]

Batches:   0%|          | 0/16 [00:00<?, ?it/s]

Split strings:   0%|          | 0/971 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/971 [00:00<?, ?it/s]

...

BM25S Retrieve:   0%|          | 0/971 [00:00<?, ?it/s]

Batches:   0%|          | 0/15 [00:00<?, ?it/s]

Split strings:   0%|          | 0/933 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/933 [00:00<?, ?it/s]

,model_key,retriever,split,corpus_field,rrf_k,corpus_encode_seconds,query_encode_seconds,Recall@1,Recall@3,Recall@5,Recall@10,Recall@20,MRR@10,nDCG@10,MAP@10
37,multilingual_e5_finetuned,hybrid_rrf,strict_val,embedding_text,30.0,34.889401,2.510894,0.337621,0.630225,0.751340,0.874598,0.938907,0.509308,0.597308,0.509308
38,multilingual_e5_finetuned,hybrid_rrf,strict_val,embedding_text,60.0,34.889401,2.510894,0.335477,0.632369,0.744909,0.873526,0.938907,0.508216,0.596246,0.508216
39,multilingual_e5_finetuned,hybrid_rrf,strict_val,embedding_text,90.0,34.889401,2.510894,0.335477,0.631297,0.744909,0.873526,0.938907,0.508027,0.596082,0.508027
27,multilingual_e5_finetuned,hybrid_rrf,strict_val,content,30.0,32.621136,2.436691,0.239014,0.563773,0.687031,0.841372,0.933548,0.425088,0.524992,0.425088
28,multilingual_e5_finetuned,hybrid_rrf,strict_val,content,60.0,32.621136,2.436691,0.235798,0.558414,0.684887,0.840300,0.933548,0.422491,0.522747,0.422491
29,multilingual_e5_finetuned,hybrid_rrf,strict_val,content,90.0,32.621136,2.436691,0.234727,0.557342,0.684887,0.840300,0.933548,0.421656,0.522107,0.421656
19,multilingual_e5_base,hybrid_rrf,strict_val,embedding_text,90.0,35.088210,2.428240,0.264737,0.551983,0.688103,0.828510,0.922830,0.438166,0.531735,0.438166
18,multilingual_e5_base,hybrid_rrf,strict_val,embedding_text,60.0,35.088210,2.428240,0.264737,0.551983,0.688103,0.828510,0.922830,0.438121,0.531692,0.438121
17,multilingual_e5_base,hybrid_rrf,strict_val,embedding_text,30.0,35.088210,2.428240,0.265809,0.554126,0.684887,0.827438,0.922830,0.438461,0.531713,0.438461
7,multilingual_e5_base,hybrid_rrf,strict_val,content,30.0,33.308657,2.421582,0.221865,0.547696,0.676313,0.827438,0.926045,0.412543,0.512153,0.412543


In [34]:
selection_split = "strict_val" if "strict_val" in selection_df["split"].unique() else "val"
ranked = selection_df[selection_df["split"].eq(selection_split)].copy()
ranked["selection_score"] = (
    .4*ranked["Recall@5"] + .3*ranked["Recall@10"]
    + .2*ranked["MRR@10"] + .1*ranked["nDCG@10"]
)
ranked = ranked.sort_values(["selection_score","Recall@10","MRR@10"], ascending=False)
best = ranked.iloc[0]
display(ranked)

,model_key,retriever,split,corpus_field,rrf_k,corpus_encode_seconds,query_encode_seconds,Recall@1,Recall@3,Recall@5,Recall@10,Recall@20,MRR@10,nDCG@10,MAP@10,selection_score
37,multilingual_e5_finetuned,hybrid_rrf,strict_val,embedding_text,30.0,34.889401,2.510894,0.337621,0.630225,0.751340,0.874598,0.938907,0.509308,0.597308,0.509308,0.724508
38,multilingual_e5_finetuned,hybrid_rrf,strict_val,embedding_text,60.0,34.889401,2.510894,0.335477,0.632369,0.744909,0.873526,0.938907,0.508216,0.596246,0.508216,0.721289
39,multilingual_e5_finetuned,hybrid_rrf,strict_val,embedding_text,90.0,34.889401,2.510894,0.335477,0.631297,0.744909,0.873526,0.938907,0.508027,0.596082,0.508027,0.721235
35,multilingual_e5_finetuned,dense,strict_val,embedding_text,NaN,34.889401,2.510894,0.294748,0.588424,0.694534,0.822079,0.894962,0.463659,0.549821,0.463659,0.672151
27,multilingual_e5_finetuned,hybrid_rrf,strict_val,content,30.0,32.621136,2.436691,0.239014,0.563773,0.687031,0.841372,0.933548,0.425088,0.524992,0.425088,0.664741
19,multilingual_e5_base,hybrid_rrf,strict_val,embedding_text,90.0,35.088210,2.428240,0.264737,0.551983,0.688103,0.828510,0.922830,0.438166,0.531735,0.438166,0.664601
18,multilingual_e5_base,hybrid_rrf,strict_val,embedding_text,60.0,35.088210,2.428240,0.264737,0.551983,0.688103,0.828510,0.922830,0.438121,0.531692,0.438121,0.664588
6,bm25,bm25,strict_val,embedding_text,NaN,0.000000,0.000000,0.304394,0.536977,0.679528,0.819936,0.905681,0.457386,0.544030,0.457386,0.663672
17,multilingual_e5_base,hybrid_rrf,strict_val,embedding_text,30.0,35.088210,2.428240,0.265809,0.554126,0.684887,0.827438,0.922830,0.438461,0.531713,0.438461,0.663050
28,multilingual_e5_finetuned,hybrid_rrf,strict_val,content,60.0,32.621136,2.436691,0.235798,0.558414,0.684887,0.840300,0.933548,0.422491,0.522747,0.422491,0.662818


In [35]:
selected_ref = TRAINED if best["model_key"] == "multilingual_e5_finetuned" else BASE
test_df = pd.DataFrame(evaluate_model(str(best["model_key"]), selected_ref, TEST_SPLITS))
test_df.to_csv(OUTPUT/"final_test_results.csv", index=False, encoding="utf-8-sig")
display(test_df.sort_values(["split","Recall@10","MRR@10"], ascending=[True,False,False]))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/33 [00:00<?, ?it/s]

Batches:   0%|          | 0/16 [00:00<?, ?it/s]

Split strings:   0%|          | 0/976 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/976 [00:00<?, ?it/s]

...

BM25S Retrieve:   0%|          | 0/976 [00:00<?, ?it/s]

Batches:   0%|          | 0/15 [00:00<?, ?it/s]

Split strings:   0%|          | 0/958 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/958 [00:00<?, ?it/s]

,model_key,retriever,split,corpus_field,rrf_k,corpus_encode_seconds,query_encode_seconds,Recall@1,Recall@3,Recall@5,Recall@10,Recall@20,MRR@10,nDCG@10,MAP@10
8,multilingual_e5_finetuned,hybrid_rrf,strict_test,content,60.0,31.904764,2.587771,0.143006,0.416493,0.603340,0.799582,0.923800,0.329020,0.440802,0.329020
9,multilingual_e5_finetuned,hybrid_rrf,strict_test,content,90.0,31.904764,2.587771,0.143006,0.417537,0.602296,0.799582,0.923800,0.328757,0.440569,0.328757
7,multilingual_e5_finetuned,hybrid_rrf,strict_test,content,30.0,31.904764,2.587771,0.141962,0.416493,0.600209,0.799582,0.923800,0.328729,0.440584,0.328729
17,multilingual_e5_finetuned,hybrid_rrf,strict_test,embedding_text,30.0,35.209854,2.521315,0.120042,0.402923,0.598121,0.795407,0.913361,0.310915,0.425963,0.310915
18,multilingual_e5_finetuned,hybrid_rrf,strict_test,embedding_text,60.0,35.209854,2.521315,0.121086,0.402923,0.594990,0.794363,0.913361,0.310960,0.425723,0.310960
19,multilingual_e5_finetuned,hybrid_rrf,strict_test,embedding_text,90.0,35.209854,2.521315,0.121086,0.401879,0.594990,0.793319,0.913361,0.310595,0.425191,0.310595
6,bm25,bm25,strict_test,embedding_text,NaN,0.000000,0.000000,0.113779,0.395616,0.585595,0.780793,0.887265,0.302271,0.415878,0.302271
16,bm25,bm25,strict_test,embedding_text,NaN,0.000000,0.000000,0.113779,0.395616,0.585595,0.780793,0.887265,0.302271,0.415878,0.302271
5,multilingual_e5_finetuned,dense,strict_test,content,NaN,31.904764,2.587771,0.157620,0.398747,0.554280,0.745303,0.873695,0.320437,0.421057,0.320437
15,multilingual_e5_finetuned,dense,strict_test,embedding_text,NaN,35.209854,2.521315,0.133612,0.378914,0.545929,0.715031,0.862213,0.299219,0.397928,0.299219


In [36]:
winner = {
    "selected_on_split": selection_split,
    "model_key": str(best["model_key"]),
    "model_path": str(TRAINED) if best["model_key"] == "multilingual_e5_finetuned" else BASE,
    "retriever": str(best["retriever"]),
    "corpus_field": str(best["corpus_field"]),
    "rrf_k": None if pd.isna(best["rrf_k"]) else int(best["rrf_k"]),
    "query_format": "query: {question}",
    "passage_format": "passage: {passage}",
    "query_instruction": "",
    "embedding_dimension": 768,
    "max_query_length": 128,
    "max_passage_length": 512,
    "metrics": {k:float(best[k]) for k in ["Recall@1","Recall@3","Recall@5","Recall@10","Recall@20","MRR@10","nDCG@10","MAP@10"]},
}
with (OUTPUT/"retriever_winner_config.json").open("w", encoding="utf-8") as f:
    json.dump(winner,f,ensure_ascii=False,indent=2)
print(json.dumps(winner,ensure_ascii=False,indent=2))

{
  "selected_on_split": "strict_val",
  "model_key": "multilingual_e5_finetuned",
  "model_path": "/content/drive/MyDrive/UIT_Legal_System/Source Code/Embedding Model/multilingual_e5_base_finetuned_ver2/checkpoints/checkpoint-484",
  "retriever": "hybrid_rrf",
  "corpus_field": "embedding_text",
  "rrf_k": 30,
  "query_format": "query: {question}",
  "passage_format": "passage: {passage}",
  "query_instruction": "",
  "embedding_dimension": 768,
  "max_query_length": 128,
  "max_passage_length": 512,
  "metrics": {
    "Recall@1": 0.33762057877813506,
    "Recall@3": 0.6302250803858521,
    "Recall@5": 0.7513397642015005,
    "Recall@10": 0.8745980707395499,
    "Recall@20": 0.9389067524115756,
    "MRR@10": 0.5093077459636945,
    "nDCG@10": 0.5973075252625629,
    "MAP@10": 0.5093077459636945
  }
}
